# Sentiment Analysis (Positive / Negative)
Trains on `IMDB_Dataset.csv`, then classifies new reviews. Run the cells top to bottom.

## 1. Upload the dataset
Select `IMDB_Dataset.csv` when the file picker appears (or drag it into the Colab Files sidebar and skip this cell).

## 2. Train and evaluate

In [3]:
import re, joblib, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

def clean(t):
    return re.sub(r"<[^>]+>", " ", t).lower()   # strip HTML tags

df = pd.read_csv("IMDB_Dataset.csv").drop_duplicates()
X, y = df.review.map(clean), (df.sentiment == "positive").astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_df=0.9, sublinear_tf=True),
    LinearSVC(C=0.5),
)
model.fit(Xtr, ytr)
pred = model.predict(Xte)
print(f"Test accuracy: {accuracy_score(yte, pred):.4f}")
print(classification_report(yte, pred, target_names=["negative", "positive"]))

Test accuracy: 0.9171
              precision    recall  f1-score   support

    negative       0.93      0.91      0.92      4940
    positive       0.91      0.93      0.92      4977

    accuracy                           0.92      9917
   macro avg       0.92      0.92      0.92      9917
weighted avg       0.92      0.92      0.92      9917



## 3. Final model
Retrain on all the data and save it.

In [4]:
model.fit(X, y)
joblib.dump(model, "sentiment_model.joblib")

['sentiment_model.joblib']

## 4. Classify new reviews

In [5]:
def classify(reviews):
    preds = model.predict([clean(r) for r in reviews])
    return ["positive" if p else "negative" for p in preds]

## 5. Aspect-based sentiment

In [6]:
#from transformers import pipeline

#sent_nn = pipeline("sentiment-analysis", truncation=True,
             #      model="distilbert-base-uncased-finetuned-sst-2-english")
#irony = pipeline("text-classification", truncation=True,
 #                model="cardiffnlp/twitter-roberta-base-irony")

#def contrast(review):
 #   parts = [p for p in re.split(r"[.!?;]", clean(review)) if p.strip()]
  #  return len({classify([p])[0] for p in parts}) == 2   # positive and negative sentences both present

#def overall(review):
 #   a = classify([review])[0]                      # your SVM
  #  b = sent_nn(review)[0]["label"].lower()        # transformer
   # ironic = irony(review)[0]["label"] == "irony"
    ##   return "negative"                          # both sarcasm signals agree
    #if "positive" in (a, b) and (ironic or contrast(review)):
     #   return "mixed / possible sarcasm"          # only one signal
    #return a if a == b else "mixed / uncertain"

In [7]:
import re, joblib, numpy as np, torch
from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.ensemble import HistGradientBoostingClassifier
from transformers import pipeline

dev = 0 if torch.cuda.is_available() else -1
sent_nn = pipeline("sentiment-analysis", truncation=True, device=dev,
                   model="distilbert-base-uncased-finetuned-sst-2-english")
irony = pipeline("text-classification", truncation=True, device=dev,
                 model="cardiffnlp/twitter-roberta-base-irony")

# Same split as section 2 (same random_state), so X_ho is truly unseen by base_model.
X_tr, X_ho, y_tr, y_ho = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
base_model = clone(model).fit(X_tr, y_tr)      # SVM trained on the 80% only

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [8]:
def chunk(text, max_words=60):
    """Split a review into ~60-word windows (tweet-sized, which is what the
    irony model was trained on) instead of truncating after the first 512 tokens."""
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", clean(text)) if s.strip()]
    out, cur, n = [], [], 0
    for s in sents:
        w = len(s.split())
        if cur and n + w > max_words:
            out.append(" ".join(cur)); cur, n = [], 0
        cur.append(s); n += w
    if cur:
        out.append(" ".join(cur))
    return out or [clean(text)]


def build_features(reviews):
    reviews = list(reviews)
    chunked = [chunk(r) for r in reviews]
    flat = [c for cs in chunked for c in cs]            # one big batch = fast on GPU

    ir = np.array([r["score"] if r["label"] == "irony" else 1 - r["score"]
                   for r in irony(flat, batch_size=64)])
    nn = np.array([r["score"] if r["label"] == "POSITIVE" else 1 - r["score"]
                   for r in sent_nn(flat, batch_size=64)])
    sv = base_model.decision_function(flat)
    full = base_model.decision_function([clean(r) for r in reviews])

    rows, i = [], 0
    for k, cs in enumerate(chunked):
        n = len(cs); sl = slice(i, i + n); i += n
        s, a, b = sv[sl], ir[sl], nn[sl]
        half = max(n // 2, 1)
        rows.append([
            full[k],                          # SVM on the whole review
            s.mean(), s.min(), s.max(),       # SVM across chunks
            s[-1], s[-min(2, n):].mean(),     # SVM on the ending (the punchline)
            s[:half].mean() - s[-half:].mean(),   # "turn": positive start -> negative end
            (s > 0).mean(),                   # fraction of positive-looking chunks
            a.max(), a.mean(), a[-1],         # irony: strongest, average, at the end
            b.mean(), b[-1], b.max() - b.min(),   # transformer: average, end, spread
            n, np.log1p(len(clean(reviews[k]).split())),
        ])
    return np.array(rows)

In [9]:
N = 1000
def chunk(text, max_words=60):
    """Split a review into ~60-word windows (tweet-sized, which is what the
    irony model was trained on) instead of truncating after the first 512 tokens."""
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", clean(text)) if s.strip()]
    out, cur, n = [], [], 0
    for s in sents:
        w = len(s.split())
        if cur and n + w > max_words:
            out.append(" ".join(cur)); cur, n = [], 0
        cur.append(s); n += w
    if cur:
        out.append(" ".join(cur))
    return out or [clean(text)]


def build_features(reviews):
    reviews = list(reviews)
    chunked = [chunk(r) for r in reviews]
    flat = [c for cs in chunked for c in cs]            # one big batch = fast on GPU

    ir = np.array([r["score"] if r["label"] == "irony" else 1 - r["score"]
                   for r in irony(flat, batch_size=64)])
    nn = np.array([r["score"] if r["label"] == "POSITIVE" else 1 - r["score"]
                   for r in sent_nn(flat, batch_size=64)])
    sv = base_model.decision_function(flat)
    full = base_model.decision_function([clean(r) for r in reviews])

    rows, i = [], 0
    for k, cs in enumerate(chunked):
        n = len(cs); sl = slice(i, i + n); i += n
        s, a, b = sv[sl], ir[sl], nn[sl]
        half = max(n // 2, 1)
        rows.append([
            full[k],                          # SVM on the whole review
            s.mean(), s.min(), s.max(),       # SVM across chunks
            s[-1], s[-min(2, n):].mean(),     # SVM on the ending (the punchline)
            s[:half].mean() - s[-half:].mean(),   # "turn": positive start -> negative end
            (s > 0).mean(),                   # fraction of positive-looking chunks
            a.max(), a.mean(), a[-1],         # irony: strongest, average, at the end
            b.mean(), b[-1], b.max() - b.min(),   # transformer: average, end, spread
            n, np.log1p(len(clean(reviews[k]).split())),
        ])
    return np.array(rows)

In [10]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_predict

N = 1000
Xm = X_ho.sample(N, random_state=0)
ym = y_ho.loc[Xm.index].values
F = build_features(Xm)

meta = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                      max_iter=200, random_state=0)
p = cross_val_predict(meta, F, ym, cv=5, method="predict_proba")[:, 1]

words = np.expm1(F[:, -1])
base_pred, meta_pred = (F[:, 0] > 0).astype(int), (p > 0.5).astype(int)
for name, mask in [("all reviews", np.ones(N, bool)),
                   ("long (>300 words)", words > 300),
                   ("SVM unsure (|score|<0.3)", np.abs(F[:, 0]) < 0.3)]:
    print(f"{name:26s} n={mask.sum():5d}  "
          f"SVM alone: {(base_pred[mask] == ym[mask]).mean():.3f}   "
          f"stacked: {(meta_pred[mask] == ym[mask]).mean():.3f}")

meta.fit(F, ym)
joblib.dump({"base": base_model, "meta": meta}, "sentiment_stack.joblib")

all reviews                n= 1000  SVM alone: 0.916   stacked: 0.925
long (>300 words)          n=  243  SVM alone: 0.942   stacked: 0.963
SVM unsure (|score|<0.3)   n=  180  SVM alone: 0.717   stacked: 0.767


['sentiment_stack.joblib']

In [11]:
def overall(review):
    """Returns (label, note). label is 'positive' / 'negative' / 'mixed / uncertain'."""
    cleaned = clean(review)
    if len(cleaned.split()) < 15:
        # The irony model is tweet-trained and over-fires on short praise
        # ("great movie" -> ironic). Don't use it on very short text.
        a = "positive" if base_model.decision_function([cleaned])[0] > 0 else "negative"
        b = sent_nn(cleaned)[0]["label"].lower()
        return (a, "") if a == b else ("mixed / uncertain", "")

    f = build_features([review])
    prob = meta.predict_proba(f)[0, 1]
    svm_label = "positive" if f[0, 0] > 0 else "negative"
    label = "positive" if prob > 0.5 else "negative"
    if abs(prob - 0.5) < 0.15:
        return "mixed / uncertain", f"P(positive)={prob:.2f}"
    note = "possible sarcasm: surface text disagrees with the stacked verdict" if label != svm_label else ""
    return label, note

## 6. Sentiment strength

In [12]:
import glob
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

!wget -nc -q https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xzf aclImdb_v1.tar.gz

texts, stars = [], []
for f in glob.glob("aclImdb/*/[pn]*/*.txt"):          # train/test, pos/neg folders
    texts.append(clean(open(f, encoding="utf8").read()))
    stars.append(int(f.split("_")[-1][:-4]))          # rating from the filename

reg = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), Ridge())
Xtr, Xte, ytr, yte = train_test_split(texts, stars, test_size=0.2, random_state=42)
reg.fit(Xtr, ytr)
print("Average error:", round(mean_absolute_error(yte, reg.predict(Xte)), 2), "stars")
reg.fit(texts, stars)                                  # retrain on everything

def rating(review):
    return int(min(max(round(reg.predict([clean(review)])[0]), 1), 10))

Average error: 1.55 stars


## 7. Emotion detection

In [13]:
emotion = pipeline("text-classification",
                   model="SamLowe/roberta-base-go_emotions", truncation=True)

POS = {"admiration", "amusement", "approval", "caring", "desire", "excitement",
       "gratitude", "joy", "love", "optimism", "pride", "relief"}
NEG = {"anger", "annoyance", "disappointment", "disgust", "embarrassment",
       "fear", "grief", "nervousness", "remorse", "sadness"}

def get_emotion(review, sentiment):
    out = emotion(review, top_k=None)                  # every emotion, best first
    out = out[0] if isinstance(out[0], list) else out
    bad = POS if sentiment == "negative" else NEG if sentiment == "positive" else set()
    return ", ".join([e["label"] for e in out if e["label"] not in bad][:2])

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

### 8. Topic modelling

In [14]:
ASPECTS = {
    "acting":   ["acting", "actor", "actress", "cast", "performance", "lead"],
    "plot":     ["plot", "story", "script", "ending", "twist"],
    "visuals":  ["visual", "effects", "cinematography", "scenery", "costume"],
    "music":    ["music", "soundtrack", "score", "songs"],
    "dialogue": ["dialogue", "writing"],
    "pacing":   ["pacing", "drags", "slow", "rushed"],
}

def aspect_sentiment(review):
    result = {}
    for part in re.split(r"[.!?;,]|\bbut\b", clean(review)):   # split into clauses
        for aspect, words in ASPECTS.items():
            if any(w in part for w in words):
                result[aspect] = classify([part])[0]
    return result

In [15]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

def show_topics(reviews, n=5):
    cv = CountVectorizer(stop_words="english", max_df=0.5, min_df=5, max_features=5000)
    lda = LatentDirichletAllocation(n_components=n, random_state=42).fit(cv.fit_transform(reviews))
    words = cv.get_feature_names_out()
    for i, t in enumerate(lda.components_):
        print(f"Topic {i+1}:", ", ".join(words[t.argsort()[-8:][::-1]]))

print("Complaints:"); show_topics(X[y == 0].sample(5000, random_state=42))
print("Praise:");     show_topics(X[y == 1].sample(5000, random_state=42))

Complaints:
Topic 1: horror, films, gore, movies, budget, low, blood, series
Topic 2: just, people, really, don, guy, time, funny, good
Topic 3: story, character, just, time, good, characters, plot, really
Topic 4: bad, just, good, movies, acting, really, don, watch
Topic 5: story, just, book, people, life, time, good, make
Praise:
Topic 1: good, horror, great, like, films, best, just, role
Topic 2: like, really, just, good, great, think, time, don
Topic 3: life, story, people, like, just, time, love, way
Topic 4: man, like, good, character, time, police, plays, just
Topic 5: story, time, great, good, best, music, films, dvd


In [ ]:
# %% Cell E: updated interactive loop (overall now returns a tuple)
while True:
    review = input("Enter a review (or type 'quit' to stop): ")
    if review.strip().lower() == "quit":
        break
    sent, note = overall(review)
    print("Overall: ", sent, f"({note})" if note else "")
    print("Rating:  ", rating(review), "/ 10")
    print("Emotion: ", get_emotion(review, sent))
    print("Aspects: ", aspect_sentiment(review) or "none found", "\n")

Enter a review (or type 'quit' to stop): The acting is superb and the lead carries every scene with real warmth. The plot is clever, the soundtrack is wonderful, and the pacing never lets up. Easily one of the best films I have seen in years.
Overall:  positive 
Rating:   10 / 10
Emotion:  admiration, approval
Aspects:  {'acting': 'positive', 'plot': 'positive', 'music': 'positive', 'pacing': 'positive'} 

Enter a review (or type 'quit' to stop): The plot is a confusing mess and the acting is wooden from start to finish. The dialogue is awkward and the pacing drags, so the film feels twice as long as it is. I regret spending my evening on this.
Overall:  negative 
Rating:   3 / 10
Emotion:  remorse, confusion
Aspects:  {'acting': 'negative', 'plot': 'negative', 'dialogue': 'negative', 'pacing': 'negative'} 

Enter a review (or type 'quit' to stop): Oh wow, what a brilliant masterpiece, I loved how the plot made no sense whatsoever. The acting was so wonderfully wooden that I kept wonde